# 1. MESA Docsynth

## Pre-requisites

- AWS credentials

## Overview

At the core of Docsynth is logic to build a prompt that will enable a foundation model to generate a synthetic representative of a document that may be in the actual corpus.
To ensure the documents cover the variety of documents found in the actual corpus, prompt construction is varied by different _profiles_ (e.g. diagnoses), matched 1:1 with each document; _styles_ (e.g. letter vs. not a letter) and _content_ (e.g. a timeline event, such as trial recruitment), selected probabilistically; _structures_ providing examples of existing document structure; and synthetic _names and locations_.

<p align="center">
  <img src="https://raw.githubusercontent.com/londonaicentre/MESA-Docsynth/refs/heads/spike/mesa-build-refactor/_assets/prompt.svg" width="700" alt="Flowchart of the document generation pipeline">
</p>

When working with Docsynth, all of these _assets_ can be customised (e.g. a new set of profiles) to generate new synthetic documents for a new domain. The library also already contains assets that can be used to generate documents for existing domains.

> 📓 "How many patients have a KRAS mutation?" is a question that a data scientist might post of a corpus of oncology notes, but one that might not be straightforward to answer. Have a look at some [example free-text](../data/sample/oncoqwen_sample_data_input.json) and you'll find that although KRAS is mentioned, in all instances it is acutally to say that KRAS is _not_ present. This would throw off keyword searches, regex and likely even NER. We will consider this problem through these notebooks. In this notebook, because this problem exists in the domain of oncology, we will select existing oncology-based assets.

## Steps

1. Import the `Generator` class from the docsynth library (which has been pre-added as a dependency to this repository), and create a reusable object from it.

In [ ]:
from docsynth.generate import Generator
generator: Generator = Generator()

2. Import some utility classes to help us work with the library.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

3. Prepare the _pipeline_ that will be used to configure document generation. This follows a set format, the structure of which is documented (as a type and as an example) in the docsynth package. An example pipeline has also been placed in this repository, and will be consumed in what follows. Take a moment to inspect [`pipeline.yml`](pipeline.yml) to understand the types of prompts we will build and thus the types of documents.

4. We have several mechanisms to generate our synthetic documents, such as different language model providers. Here, we will use AWS Bedrock batch processing, which leverages the credentials we already have in place. Run the following code to start batch generation.

>❗Batch inference can be an expensive process. If you would prefer to skip this step, instead run `mv .job_id.example.json .job_id.json`.

In [ ]:
generator.generate_via_batch(os.environ["BUCKET"], os.environ["BEDROCK_EXECUTION_ROLE"])

5. (Optional) Inspect `anthropic_batch_job.jsonl` to see the prompts we are sending to AWS Bedrock (Claude Sonnet) to generate our documents.

6. Because this is a batch job, we may need to wait before we can process the outputs. We can check the status of the job by running the command below.

In [ ]:
generator.check_batch_output_status(os.environ["BUCKET"])

7. Once the above check confirms our batch output is ready, we can download it, parse it, output it in a format for the rest of the pipeline, and upload it.

In [ ]:
generator.extract_batch_output(os.environ["BUCKET"])

8. (Optional) Inspect `anthropic_batch_job.jsonl.out` to see the raw responses from the LLM.

9. With our synthetic documents uploaded, we can move to the [next notebook](../datagen/datagen.ipynb).